In [1]:
# libraries
!pip install -q --upgrade \
    datasets \
    transformers \
    evaluate \
    torch torchvision timm

In [ ]:
# unzip the dataset
import os
import zipfile

zip_path = "/Users/skim/Documents/cs231n/finalproject/GeoGuessrCV/archive.zip"
extract_dir = "./geo50k"
os.makedirs(extract_dir, exist_ok=True)

with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(extract_dir)

print(f"Unzipped {zip_path} → {extract_dir}")

Unzipped /Users/skim/Documents/cs231n/finalproject/GeoGuessrCV/archive.zip → ./geo50k


In [3]:
# If you don't already have them:
!pip install -q pandas scikit-learn


In [19]:
import os
import glob
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

# 1) Point to your unzipped folder (modify if needed)
DATA_DIR = "/Users/skim/Documents/cs231n/finalproject/GeoGuessrCV/geo50k/compressed_dataset"
assert os.path.isdir(DATA_DIR), f"Folder not found: {DATA_DIR}"


In [20]:
all_entries = [] 

for country in os.listdir(DATA_DIR):
    country_dir = os.path.join(DATA_DIR, country)
    if not os.path.isdir(country_dir):
        continue

    # You can add other extensions if needed (e.g. "*.png")
    pattern = os.path.join(country_dir, "*.jpg")
    file_list = glob.glob(pattern)

    for fp in file_list:
        all_entries.append((fp, country))

print(f"✔️ Found {len(all_entries):,} images across {len(os.listdir(DATA_DIR))} country‐folders.")


✔️ Found 49,997 images across 125 country‐folders.


In [23]:
df = pd.DataFrame(all_entries, columns=["file_path", "label"])

# 2a) Total counts
print("Total images:", len(df))
print("Unique countries:", df["label"].nunique())

# 2b) Class distribution
country_counts = df["label"].value_counts()
print("\nSample country counts:")
print(country_counts.head(10))

# 2c) Check for missing files
missing = [fp for fp in df["file_path"] if not os.path.isfile(fp)]
assert len(missing) == 0, f"Missing files: {missing[:5]}"


Total images: 49997
Unique countries: 124

Sample country counts:
label
United States     12014
Japan              3840
France             3573
United Kingdom     2484
Brazil             2320
Russia             1761
Australia          1704
Canada             1382
South Africa       1183
Spain              1075
Name: count, dtype: int64


In [28]:
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

# 1) Count how many images per country
country_counts = Counter(df["label"])

# 2) Separate out singleton‐labels vs. multi‐sample labels
singletons = [row for row in df.itertuples(index=False) if country_counts[row.label] == 1]
multi_df   = df[df["label"].isin([c for c, cnt in country_counts.items() if cnt >= 2])].reset_index(drop=True)

print("Number of singleton‐countries:", len(singletons))      # e.g., classes with count=1
print("Remaining for stratified split:", len(multi_df), "examples")


# 3) Stratified split on the “multi” subset
train_multi, hold_multi = train_test_split(
    multi_df,
    test_size=0.20,
    random_state=42,
    stratify=multi_df["label"]
)

# 4) Put *all* singletons into the TRAIN set (or distribute as you wish)
#    Here we simply append them to train_multi
singletons_df = pd.DataFrame(singletons, columns=["file_path", "label"])
train_df = pd.concat([train_multi, singletons_df], ignore_index=True)
hold_df  = hold_multi.copy()

print(f"After merging singletons → train:")
print(f"  Train set size: {len(train_df):,}")
print(f"  Hold‐out set size: {len(hold_df):,}")


Number of singleton‐countries: 13
Remaining for stratified split: 49984 examples
After merging singletons → train:
  Train set size: 40,000
  Hold‐out set size: 9,997


In [29]:
val_df, test_df = train_test_split(
    hold_df,
    test_size=0.50,
    random_state=42,
    shuffle=True
)

print(f"Validation set: {len(val_df):,} images  (≈10%)")
print(f"Test       set: {len(test_df):,} images  (≈10%)")


Validation set: 4,998 images  (≈10%)
Test       set: 4,999 images  (≈10%)


In [30]:
def summary(split_df, name):
    cts = split_df["label"].value_counts()
    print(f"→ {name}: {len(split_df):,} images, {cts.shape[0]} countries")
    print(f"  Min per‐country: {cts.min()}, Max per‐country: {cts.max()}\n")

summary(train_df, "Train")
summary(val_df,   "Validation")
summary(test_df,  "Test")


→ Train: 40,000 images, 124 countries
  Min per‐country: 1, Max per‐country: 9611

→ Validation: 4,998 images, 104 countries
  Min per‐country: 1, Max per‐country: 1198

→ Test: 4,999 images, 101 countries
  Min per‐country: 1, Max per‐country: 1205



In [31]:
os.makedirs("splits", exist_ok=True)

# 6a) Save train/val/test CSVs
train_csv = "splits/train.csv"
val_csv   = "splits/val.csv"
test_csv  = "splits/test.csv"

train_df.to_csv(train_csv, index=False)
val_df.to_csv(val_csv,     index=False)
test_df.to_csv(test_csv,   index=False)

# 6b) Build and save label2id.csv
labels = sorted(df["label"].unique())
label2id = {lab: i for i, lab in enumerate(labels)}
pd.DataFrame({"country": labels, "id": list(label2id.values())}) \
  .to_csv("splits/label2id.csv", index=False)

print("Saved:")
print("  -", train_csv)
print("  -", val_csv)
print("  -", test_csv)
print("  - splits/label2id.csv")


Saved:
  - splits/train.csv
  - splits/val.csv
  - splits/test.csv
  - splits/label2id.csv
